# LeetCode 452 — Minimum Number of Arrows to Burst Balloons bằng Python

Link đề: https://leetcode.com/problems/minimum-number-of-arrows-to-burst-balloons/ · Độ khó: **Medium** · Chủ đề: Greedy, Array, Sorting, Intervals

Skeleton LeetCode:

```python
class Solution:
    def findMinArrowShots(self, points: List[List[int]]) -> int:
        ...
```

Ghi chú: đề được viết lại theo hiểu biết về đề gốc; nên đối chiếu lại đề gốc nếu muốn khớp từng chữ. Mọi đáp án trong notebook được kiểm chứng bằng code, so với một **oracle vét cạn độc lập** (thử mọi tập mũi tên) trên input nhỏ.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: cho các "quả bóng" mỗi quả là một **đoạn** `[x_start, x_end]` trên trục x; một mũi tên bắn thẳng đứng tại hoành độ `x` làm nổ **mọi** quả bóng có `x_start <= x <= x_end`. Tìm **số mũi tên ít nhất** để nổ hết.
2. Nắm các công cụ Python: danh sách các cặp `[a, b]`, `sorted(..., key=lambda ...)`, `float("-inf")`, `itertools.combinations` (để vét cạn), `bisect` (tìm nhị phân), và kiểm tra điều kiện **đoạn cắt nhau** (giao nhau tại ít nhất một điểm).
3. Đi từ **vét cạn mọi tập mũi tên** → **tham lam: sắp theo đầu phải** → **tham lam: sắp theo đầu trái, giữ giao của nhóm** → **quét đường thẳng (sweep line)** → **đối ngẫu: tối đa các đoạn rời nhau (DP + bisect)**.
4. Hiểu mấu chốt: bắn mũi tên ngay tại **đầu phải** của quả bóng "kết thúc sớm nhất" là lựa chọn không bao giờ tệ (lập luận hoán đổi), nên chỉ cần **sắp theo đầu phải** và quét một lần.
5. Tránh các lỗi kinh điển: dùng `>=` thay cho `>` khi hai đoạn chỉ chạm nhau tại một điểm, khởi tạo `last = 0` (sai với toạ độ âm), sắp theo đầu trái rồi không thu hẹp giao, so sánh `a[1] - b[1]` bị tràn số ở ngôn ngữ khác, và tham lam kiểu "bắn chỗ nhiều bóng nhất" (sai).

## 2. Đề bài & ví dụ

Có một số quả bóng hình cầu dán trên một bức tường phẳng, được xem là mặt phẳng XY. Các quả bóng được cho bởi mảng 2 chiều `points`, trong đó `points[i] = [x_start, x_end]` là quả bóng có **đường kính nằm ngang** trải từ `x_start` đến `x_end`. Bạn **không biết toạ độ y**.

Có thể bắn các mũi tên **thẳng đứng lên** (chiều dương của y) từ các điểm khác nhau trên trục x. Một quả bóng `[x_start, x_end]` bị nổ bởi mũi tên bắn tại `x` nếu `x_start <= x <= x_end`. **Không giới hạn** số mũi tên, và mũi tên bay lên mãi mãi, nổ mọi quả bóng trên đường đi.

Trả về **số mũi tên tối thiểu** cần bắn để nổ hết các quả bóng.

**Ví dụ 1**

- Input: `points = [[10,16],[2,8],[1,6],[7,12]]`
- Output: `2`
- Giải thích: bắn tại `x = 6` nổ `[2,8]` và `[1,6]`; bắn tại `x = 11` nổ `[10,16]` và `[7,12]`.

**Ví dụ 2**

- Input: `points = [[1,2],[3,4],[5,6],[7,8]]`
- Output: `4`
- Giải thích: mỗi quả cần một mũi tên riêng.

**Ví dụ 3**

- Input: `points = [[1,2],[2,3],[3,4],[4,5]]`
- Output: `2`
- Giải thích: bắn tại `x = 2` nổ `[1,2]` và `[2,3]`; bắn tại `x = 4` nổ `[3,4]` và `[4,5]`.

**Ràng buộc:**

- `1 <= points.length <= 10^5`
- `points[i].length == 2`
- `-2^31 <= x_start < x_end <= 2^31 - 1`

Chú ý: các đoạn là **đoạn đóng**: hai quả `[1,2]` và `[2,3]` **chạm nhau** tại `x = 2` nên **một** mũi tên nổ được cả hai (Ví dụ 3).

In [1]:
from typing import List
from itertools import combinations
from bisect import bisect_left
import random
import time


def covers_all(arrows, points):
    """Kiểm tra tập mũi tên `arrows` (các hoành độ) có nổ hết mọi quả bóng không."""
    return all(any(a <= x <= b for x in arrows) for a, b in points)


examples = [
    ([[10, 16], [2, 8], [1, 6], [7, 12]], 2),
    ([[1, 2], [3, 4], [5, 6], [7, 8]], 4),
    ([[1, 2], [2, 3], [3, 4], [4, 5]], 2),
]
# kiểm tra các cách bắn nêu trong đề
assert covers_all([6, 11], examples[0][0])
assert covers_all([2, 4], examples[2][0])
assert not covers_all([6], examples[0][0])
for pts, expected in examples:
    print(pts, "->", expected)

[[10, 16], [2, 8], [1, 6], [7, 12]] -> 2
[[1, 2], [3, 4], [5, 6], [7, 8]] -> 4
[[1, 2], [2, 3], [3, 4], [4, 5]] -> 2


## 3. Kiến thức nền

### 3.1. Đoạn đóng và điều kiện "bị nổ"

Quả bóng `[a, b]` bị mũi tên ở `x` nổ khi `a <= x <= b` (cả hai đầu **bao gồm**). Python cho phép viết **so sánh dây chuyền** `a <= x <= b`, ngắn gọn và đúng nghĩa toán học.

In [2]:
a, b = 2, 8
for x in (1, 2, 5, 8, 9):
    print(f"x = {x}: {a} <= {x} <= {b} ->", a <= x <= b)

x = 1: 2 <= 1 <= 8 -> False
x = 2: 2 <= 2 <= 8 -> True
x = 5: 2 <= 5 <= 8 -> True
x = 8: 2 <= 8 <= 8 -> True
x = 9: 2 <= 9 <= 8 -> False


### 3.2. Hai đoạn có chung điểm không?

Hai đoạn `[a1, b1]` và `[a2, b2]` **có chung điểm** khi `max(a1, a2) <= min(b1, b2)`. Cụ thể, nếu `a2 <= b1` (đoạn thứ hai bắt đầu trước hoặc đúng lúc đoạn thứ nhất kết thúc) và đoạn thứ nhất bắt đầu trước đoạn thứ hai. Với đoạn đóng, **chạm nhau tại một điểm cũng tính là có chung điểm**.

In [3]:
def share_point(p, q):
    return max(p[0], q[0]) <= min(p[1], q[1])


print(share_point([1, 6], [2, 8]))    # True: chung [2, 6]
print(share_point([1, 2], [2, 3]))    # True: chạm nhau tại 2
print(share_point([1, 2], [3, 4]))    # False

True
True
False


### 3.3. Sắp xếp danh sách các cặp bằng `key`

`sorted(points, key=lambda p: p[1])` sắp các đoạn theo **đầu phải** `p[1]`. `lambda p: p[1]` là hàm vô danh trả về phần tử thứ hai của cặp. `sorted` **không sửa** list gốc (còn `points.sort(...)` thì sửa tại chỗ). Sắp xếp mặc định `sorted(points)` so theo `p[0]` rồi tới `p[1]`.

In [4]:
pts = [[10, 16], [2, 8], [1, 6], [7, 12]]
print(sorted(pts, key=lambda p: p[1]))     # theo đầu phải: [[1,6],[2,8],[7,12],[10,16]]
print(sorted(pts))                         # theo đầu trái (mặc định)
print(pts)                                 # list gốc không đổi

[[1, 6], [2, 8], [7, 12], [10, 16]]
[[1, 6], [2, 8], [7, 12], [10, 16]]
[[10, 16], [2, 8], [1, 6], [7, 12]]


### 3.4. Giá trị "vô cực âm" `float("-inf")`

Để khởi tạo "vị trí mũi tên gần nhất" khi **chưa bắn mũi nào**, ta cần một giá trị **nhỏ hơn mọi toạ độ**: `float("-inf")`. Không dùng `0` vì toạ độ có thể âm (đến `-2^31`).

In [5]:
last = float("-inf")
print(last < -2**31, last < 0)      # True True
print(-5 > 0, -5 > last)            # False True : -5 > 0 sai, nhưng -5 > -inf đúng

True True
False True


### 3.5. `itertools.combinations` — vét cạn mọi cách chọn

`combinations(seq, k)` sinh mọi tập con có `k` phần tử (không lặp, không phân biệt thứ tự). Ta dùng để vét cạn: "thử mọi bộ `k` điểm bắn".

In [6]:
print(list(combinations([1, 2, 3, 4], 2)))        # 6 tập con
print(len(list(combinations(range(10), 3))))      # 120 = C(10,3)

[(1, 2), (1, 3), (1, 4), (2, 3), (2, 4), (3, 4)]
120


### 3.6. `bisect_left` — tìm nhị phân trong list đã sắp xếp

`bisect_left(arr, v)` trả về số phần tử của `arr` **nhỏ hơn** `v` (vị trí chèn bên trái), trong `O(log n)`. Ta sẽ dùng cho Cách 5 để đếm "có bao nhiêu đoạn kết thúc **trước** điểm `v`".

In [7]:
ends = [3, 5, 5, 8, 12]
print(bisect_left(ends, 5))     # 1: chỉ có 3 nhỏ hơn 5
print(bisect_left(ends, 6))     # 3: 3, 5, 5 nhỏ hơn 6
print(bisect_left(ends, 100))   # 5

1
3
5


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Xét Ví dụ 1: `[[10,16],[2,8],[1,6],[7,12]]`. Vẽ trục số:

```
x:   1 2 3 4 5 6 7 8 9 10 11 12 13 14 15 16
[1,6]    ───────────
[2,8]      ─────────────
[7,12]               ─────────────
[10,16]                    ──────────────────
```

Câu hỏi: **mũi tên đầu tiên nên bắn ở đâu?** Hãy nhìn quả bóng **kết thúc sớm nhất**: `[1,6]` (đầu phải `6` nhỏ nhất). Quả này **bắt buộc** bị một mũi tên nổ; mũi tên đó phải nằm trong `[1,6]`. Trong các vị trí có thể, **càng về bên phải càng tốt** (vì các quả bóng khác không còn xuất hiện ở bên trái `1..6` mà chưa nổ, ta chỉ có lợi nếu dịch sang phải để chạm thêm quả bóng bắt đầu muộn hơn). Vị trí tốt nhất là **đầu phải `6`**.

- Bắn ở `6`: nổ `[1,6]` và `[2,8]` (vì `2 <= 6 <= 8`). `[7,12]` có `7 > 6` nên không nổ.
- Còn lại `[7,12]`, `[10,16]`. Quả kết thúc sớm nhất là `[7,12]` → bắn ở `12`: nổ `[7,12]` và `[10,16]`.

Tổng cộng **2** mũi tên — đúng đáp án. Quy trình: **sắp theo đầu phải; bắn tại đầu phải của quả đầu tiên chưa nổ; bỏ qua mọi quả có đầu trái `<=` vị trí vừa bắn; lặp lại.** Ta kiểm chứng bằng code nhỏ:

In [8]:
pts = sorted([[10, 16], [2, 8], [1, 6], [7, 12]], key=lambda p: p[1])
print("sau khi sắp theo đầu phải:", pts)
arrows = []
for a, b in pts:
    if not arrows or a > arrows[-1]:       # chưa có mũi tên nào chạm tới quả này
        arrows.append(b)                   # bắn tại đầu phải
print("các mũi tên:", arrows, "| nổ hết:", covers_all(arrows, pts))
assert arrows == [6, 12] and covers_all(arrows, pts)

sau khi sắp theo đầu phải: [[1, 6], [2, 8], [7, 12], [10, 16]]
các mũi tên: [6, 12] | nổ hết: True


## 5. Cách 1 — Brute Force / baseline: thử mọi tập mũi tên, từ ít đến nhiều

**Ý tưởng.** Chưa dùng trực giác nào. Thử `k = 1, 2, 3, ...`: với mỗi `k`, thử **mọi tập `k` hoành độ** và kiểm tra có nổ hết mọi quả không; `k` nhỏ nhất thoả là đáp án.

Hoành độ bắn có vô số, nhưng chỉ cần thử các **đầu mút** của các đoạn: nếu một tập mũi tên nổ hết, ta dịch mỗi mũi tên sang phải đến đầu phải gần nhất (không rời khỏi bất kỳ quả nào đang chạm) mà vẫn nổ hết — nên tập ứng viên là mọi `x_start`, `x_end`. Bắn tại các đầu phải là đủ, nhưng ở baseline ta dùng **cả hai loại đầu mút** để không phụ thuộc vào kết luận của lời giải sau.

**Nút thắt.** Có `2n` ứng viên và phải thử `C(2n, k)` tập: bùng nổ tổ hợp; chỉ dùng được với `n` cỡ ≤ 10. Nhưng nó là **oracle độc lập** tuyệt vời để kiểm chứng.

In [9]:
def min_arrows_bruteforce(points):
    candidates = sorted({x for a, b in points for x in (a, b)})
    for k in range(1, len(points) + 1):
        for combo in combinations(candidates, k):
            if covers_all(combo, points):
                return k


for pts, expected in examples:
    r = min_arrows_bruteforce(pts)
    assert r == expected
    print(pts, "->", r)

[[10, 16], [2, 8], [1, 6], [7, 12]] -> 2
[[1, 2], [3, 4], [5, 6], [7, 8]] -> 4
[[1, 2], [2, 3], [3, 4], [4, 5]] -> 2


**Độ phức tạp.** Thời gian lên tới `O(C(2n, k) · n · k)` — mũ; bộ nhớ `O(n)`. Chỉ dùng làm oracle với `n ≤ 8`.

## 6. Ý tưởng thuật toán / bước đột phá

**Bước đột phá — lập luận hoán đổi (exchange argument):**

Gọi `E` là quả bóng có **đầu phải nhỏ nhất** (`x_end` nhỏ nhất), có đầu phải là `e`. Mọi lời giải đúng phải có ít nhất một mũi tên `t` nằm trong `E`, tức `E.start <= t <= e`. Thay mũi tên `t` bằng mũi tên đặt tại **`e`**: mọi quả bóng mà `t` từng nổ vẫn bị nổ bởi `e`. Vì sao?

- Quả bóng `B = [a, b]` bị nổ bởi `t` nghĩa là `a <= t <= b`.
- Vì `e` là đầu phải **nhỏ nhất** nên `b >= e` (mọi đầu phải `>= e`), và `t <= e`. Ta có `a <= t <= e <= b` → `a <= e <= b`: `B` bị nổ bởi `e`. ∎

Vậy **tồn tại lời giải tối ưu có mũi tên ở `e`**. Bỏ mọi quả bị nổ bởi `e` rồi lặp lại cho phần còn lại (bài toán con cùng dạng). Đây chính là thuật toán tham lam: **sắp theo đầu phải, bắn ở đầu phải của quả đầu tiên chưa nổ**.

Cụ thể khi đã sắp theo đầu phải và nhớ `last` = vị trí mũi tên gần nhất: quả `[a, b]` **chưa bị nổ** khi `a > last` (vì mọi quả đang xét có `b >= last`, nên nổ ⇔ `a <= last`). Hãy kiểm chứng nhận xét "bắn ở đầu phải không tệ hơn bắn ở nơi khác trong quả đó" bằng một thí nghiệm nhỏ:

In [10]:
rng = random.Random(452)
for _ in range(500):
    n = rng.randint(1, 6)
    pts = []
    for _ in range(n):
        a = rng.randint(-5, 8)
        pts.append([a, a + rng.randint(1, 6)])
    E = min(pts, key=lambda p: p[1])
    e = E[1]
    # bất kỳ vị trí t trong E: tập quả bị nổ bởi t là con của tập quả bị nổ bởi e
    for t in range(E[0], E[1] + 1):
        hit_t = {i for i, (a, b) in enumerate(pts) if a <= t <= b}
        hit_e = {i for i, (a, b) in enumerate(pts) if a <= e <= b}
        assert hit_t <= hit_e
print("Thí nghiệm: bắn ở đầu phải của quả kết thúc sớm nhất luôn nổ nhiều không kém mọi vị trí khác trong quả đó.")

Thí nghiệm: bắn ở đầu phải của quả kết thúc sớm nhất luôn nổ nhiều không kém mọi vị trí khác trong quả đó.


## 7. Các cách tiếp theo

### Cách 2 — Tham lam: **sắp theo đầu phải** (cách chuẩn)

**Giải thích từng đoạn.**

1. `points.sort(key=lambda p: p[1])` (hoặc `sorted`): sắp theo đầu phải tăng dần.
2. `ans = 0`, `last = float("-inf")` (chưa bắn mũi nào).
3. Với mỗi `[a, b]`: nếu `a > last` (mũi tên gần nhất **không** chạm tới quả này) → bắn mũi mới: `ans += 1`, `last = b` (tại đầu phải).
4. Trả `ans`.

Điều kiện dùng `>` **nghiêm ngặt**: khi `a == last` hai đoạn chạm nhau, mũi tên tại `last` vẫn nổ được.

In [11]:
def arrows_by_end(points):
    ans, last = 0, float("-inf")
    for a, b in sorted(points, key=lambda p: p[1]):
        if a > last:
            ans += 1
            last = b
    return ans


for pts, expected in examples:
    r = arrows_by_end(pts)
    assert r == expected
    print(pts, "->", r)

[[10, 16], [2, 8], [1, 6], [7, 12]] -> 2
[[1, 2], [3, 4], [5, 6], [7, 8]] -> 4
[[1, 2], [2, 3], [3, 4], [4, 5]] -> 2


**Code nộp LeetCode (Cách 2):**

```python
class Solution:
    def findMinArrowShots(self, points: List[List[int]]) -> int:
        ans, last = 0, float("-inf")
        for a, b in sorted(points, key=lambda p: p[1]):
            if a > last:
                ans += 1
                last = b
        return ans
```

**Độ phức tạp.** Sắp xếp `O(n log n)` + quét `O(n)` → `O(n log n)` thời gian; bộ nhớ `O(n)` (danh sách đã sắp; `O(1)` thêm nếu sắp tại chỗ).

### Cách 3 — Tham lam: **sắp theo đầu trái**, giữ **giao** của nhóm đang xét

**Ý tưởng khác.** Sắp theo đầu **trái**. Ta gom các quả bóng liên tiếp thành **nhóm**: một nhóm là tập các quả có **chung một điểm**, tức giao của chúng khác rỗng. Giữ giao `[lo, hi]` của nhóm hiện tại (`hi` = đầu phải **nhỏ nhất** trong nhóm, `lo` = đầu trái lớn nhất). Với quả `[a, b]` kế tiếp:

- Nếu `a <= hi`: vẫn chung điểm với cả nhóm → **thu hẹp giao**: `lo = max(lo, a) = a` (vì sắp theo đầu trái), `hi = min(hi, b)`.
- Nếu `a > hi`: không giao với nhóm → **nhóm mới**, `ans += 1`, `[lo, hi] = [a, b]`.

Mỗi nhóm cần đúng **một** mũi tên (bắn ở bất kỳ điểm nào trong giao). Chú ý: phải **thu hẹp `hi = min(hi, b)`**; nếu quên, nhóm "giãn" ra và sai (lỗi 1 ở mục 13).

In [12]:
def arrows_by_start(points):
    pts = sorted(points)                    # theo đầu trái (rồi đầu phải)
    ans = 0
    lo, hi = None, float("-inf")
    for a, b in pts:
        if a > hi:                          # không giao với nhóm hiện tại -> nhóm mới
            ans += 1
            lo, hi = a, b
        else:                               # thu hẹp giao của nhóm
            lo = max(lo, a)
            hi = min(hi, b)
    return ans


for pts, expected in examples:
    r = arrows_by_start(pts)
    assert r == expected
    print(pts, "->", r)

[[10, 16], [2, 8], [1, 6], [7, 12]] -> 2
[[1, 2], [3, 4], [5, 6], [7, 8]] -> 4
[[1, 2], [2, 3], [3, 4], [4, 5]] -> 2


**Độ phức tạp.** `O(n log n)` thời gian, `O(n)` bộ nhớ (như Cách 2).

### Cách 4 — **Quét đường thẳng (sweep line)** theo các sự kiện

**Ý tưởng khác.** Biến mỗi đoạn thành **hai sự kiện**: `(a, bắt đầu)` và `(b, kết thúc)`. Sắp mọi sự kiện theo hoành độ; **cùng hoành độ thì sự kiện "bắt đầu" đứng trước "kết thúc"** (vì đoạn đóng: một quả bắt đầu đúng tại `x` vẫn bị mũi tên bắn ở `x` nổ). Quét từ trái sang phải, giữ tập `active` các quả đã bắt đầu mà chưa nổ:

- Gặp **bắt đầu**: thêm quả vào `active`.
- Gặp **kết thúc** của quả `i`: nếu `i` còn trong `active` (chưa nổ) thì **buộc** phải bắn ngay tại hoành độ này (đây là hạn chót, bắn trễ hơn sẽ trượt quả này): `ans += 1`, mũi tên nổ **mọi quả trong `active`**, xoá sạch `active`. Nếu `i` không còn trong `active` thì nó đã nổ rồi, bỏ qua.

**Giải thích từng đoạn.** `events = []`; với mỗi `i, (a, b)`: `events.append((a, 0, i))`, `events.append((b, 1, i))` (kiểu `0` = bắt đầu, `1` = kết thúc nên sắp xếp tuple đặt bắt đầu trước); `events.sort()`.

In [13]:
def arrows_sweep(points):
    events = []
    for i, (a, b) in enumerate(points):
        events.append((a, 0, i))            # 0 = bắt đầu (đứng trước 1 khi cùng hoành độ)
        events.append((b, 1, i))            # 1 = kết thúc
    events.sort()
    active = set()
    ans = 0
    for x, kind, i in events:
        if kind == 0:
            active.add(i)
        elif i in active:                   # quả i sắp "hết hạn" mà chưa nổ -> phải bắn tại x
            ans += 1
            active.clear()                  # mũi tên tại x nổ mọi quả đang mở
    return ans


for pts, expected in examples:
    r = arrows_sweep(pts)
    assert r == expected
    print(pts, "->", r)

[[10, 16], [2, 8], [1, 6], [7, 12]] -> 2
[[1, 2], [3, 4], [5, 6], [7, 8]] -> 4
[[1, 2], [2, 3], [3, 4], [4, 5]] -> 2


**Độ phức tạp.** Sắp `2n` sự kiện `O(n log n)`; quét `O(n)` (mỗi quả vào/ra `active` tối đa một lần, `clear` được tính theo số phần tử đã thêm) → `O(n log n)` thời gian, `O(n)` bộ nhớ.

### Cách 5 — **Đối ngẫu**: số mũi tên tối thiểu = số quả bóng **đôi một không chung điểm** tối đa (DP + `bisect`)

**Ý tưởng khác hẳn.** Nếu có `m` quả bóng **đôi một không chung điểm** thì cần **ít nhất `m`** mũi tên (không mũi tên nào nổ được hai quả trong số đó). Với các **đoạn trên trục số**, kết quả đẹp: số mũi tên tối thiểu **bằng đúng** `m` lớn nhất (tính chất đối ngẫu của đồ thị khoảng). Do đó ta có thể tính `m` bằng **quy hoạch động**, không cần nghĩ tới mũi tên:

1. Sắp các quả theo đầu phải: `ends` = danh sách các đầu phải đã sắp.
2. `dp[i]` = số quả rời nhau tối đa trong `i` quả đầu tiên (đã sắp).
3. Với quả thứ `i` (1-indexed) là `[a, b]`: có thể **không chọn** → `dp[i-1]`; hoặc **chọn** quả này, thì mọi quả đã chọn trước phải kết thúc **trước `a`** (đầu phải `< a`): có `j = bisect_left(ends, a)` quả như vậy → `1 + dp[j]`.
4. `dp[i] = max(dp[i-1], 1 + dp[j])`. Đáp án `dp[n]`.

Đây là thuật toán "lập lịch khoảng có trọng số" giản lược; nó cho kết quả **trùng** Cách 2 nhờ đối ngẫu (ta sẽ kiểm chứng bằng `assert`).

In [14]:
def arrows_by_duality_dp(points):
    pts = sorted(points, key=lambda p: p[1])
    ends = [b for _, b in pts]
    n = len(pts)
    dp = [0] * (n + 1)
    for i in range(1, n + 1):
        a, b = pts[i - 1]
        j = bisect_left(ends, a)            # số quả có đầu phải < a  (đều đứng trước quả i)
        dp[i] = max(dp[i - 1], 1 + dp[j])
    return dp[n]


for pts, expected in examples:
    r = arrows_by_duality_dp(pts)
    assert r == expected
    print(pts, "->", r)

[[10, 16], [2, 8], [1, 6], [7, 12]] -> 2
[[1, 2], [3, 4], [5, 6], [7, 8]] -> 4
[[1, 2], [2, 3], [3, 4], [4, 5]] -> 2


**Độ phức tạp.** Sắp xếp `O(n log n)`; mỗi `bisect_left` `O(log n)` → tổng `O(n log n)` thời gian; bộ nhớ `O(n)` cho `dp` và `ends`.

Cách 5 không nhanh hơn, nhưng cho góc nhìn quan trọng: bài này là "đối ngẫu" của **Non-overlapping Intervals** (435): `435 = n − (số quả rời nhau tối đa)` còn `452 =` số quả rời nhau tối đa (với quy ước "chạm nhau là có chung điểm").

## 8. TRACE từng bước

Ta trace **Cách 2** (sắp theo đầu phải) và **Cách 4** (sweep) trên `[[10,16],[2,8],[1,6],[7,12]]`.

In [15]:
def trace_by_end(points):
    pts = sorted(points, key=lambda p: p[1])
    print("sau khi sắp theo đầu phải:", pts)
    ans, last = 0, float("-inf")
    for a, b in pts:
        if a > last:
            ans += 1
            print(f"[{a:>2},{b:>2}]: a={a} > last={last} -> BẮN mũi thứ {ans} tại x={b}")
            last = b
        else:
            print(f"[{a:>2},{b:>2}]: a={a} <= last={last} -> đã nổ bởi mũi tên tại {last}")
    return ans


assert trace_by_end([[10, 16], [2, 8], [1, 6], [7, 12]]) == 2

sau khi sắp theo đầu phải: [[1, 6], [2, 8], [7, 12], [10, 16]]
[ 1, 6]: a=1 > last=-inf -> BẮN mũi thứ 1 tại x=6
[ 2, 8]: a=2 <= last=6 -> đã nổ bởi mũi tên tại 6
[ 7,12]: a=7 > last=6 -> BẮN mũi thứ 2 tại x=12
[10,16]: a=10 <= last=12 -> đã nổ bởi mũi tên tại 12


In [16]:
def trace_sweep(points):
    events = []
    for i, (a, b) in enumerate(points):
        events.append((a, 0, i))
        events.append((b, 1, i))
    events.sort()
    active, ans = set(), 0
    for x, kind, i in events:
        name = "bắt đầu" if kind == 0 else "kết thúc"
        if kind == 0:
            active.add(i)
            print(f"x={x:>2} {name} quả {i} {points[i]} -> active={sorted(active)}")
        elif i in active:
            ans += 1
            print(f"x={x:>2} {name} quả {i} {points[i]} -> chưa nổ: BẮN tại {x}, nổ {sorted(active)}, ans={ans}")
            active.clear()
        else:
            print(f"x={x:>2} {name} quả {i} {points[i]} -> đã nổ trước đó, bỏ qua")
    return ans


assert trace_sweep([[10, 16], [2, 8], [1, 6], [7, 12]]) == 2

x= 1 bắt đầu quả 2 [1, 6] -> active=[2]
x= 2 bắt đầu quả 1 [2, 8] -> active=[1, 2]
x= 6 kết thúc quả 2 [1, 6] -> chưa nổ: BẮN tại 6, nổ [1, 2], ans=1
x= 7 bắt đầu quả 3 [7, 12] -> active=[3]
x= 8 kết thúc quả 1 [2, 8] -> đã nổ trước đó, bỏ qua
x=10 bắt đầu quả 0 [10, 16] -> active=[0, 3]
x=12 kết thúc quả 3 [7, 12] -> chưa nổ: BẮN tại 12, nổ [0, 3], ans=2
x=16 kết thúc quả 0 [10, 16] -> đã nổ trước đó, bỏ qua


Bảng trace Cách 2 trên `[[10,16],[2,8],[1,6],[7,12]]` (đã sắp theo đầu phải: `[1,6],[2,8],[7,12],[10,16]`):

| Quả `[a,b]` | `last` trước | `a > last`? | Hành động | `last` sau | `ans` |
|---|---|---|---|---|---|
| `[1,6]` | `-inf` | có | bắn tại `6` | 6 | 1 |
| `[2,8]` | 6 | không (`2 <= 6`) | đã nổ | 6 | 1 |
| `[7,12]` | 6 | có (`7 > 6`) | bắn tại `12` | 12 | 2 |
| `[10,16]` | 12 | không (`10 <= 12`) | đã nổ | 12 | 2 |

Kết quả `2`. Nhận xét: `last` luôn là hoành độ mũi tên **gần nhất bên phải**; vì các quả được duyệt theo đầu phải tăng dần nên mọi quả còn lại có đầu phải `>= last`, và chúng bị nổ **chỉ cần** `a <= last`.

## 9. Code nộp LeetCode

Bản nộp dùng **Cách 2**: sắp theo đầu phải rồi quét một lần. Dùng `float("-inf")` để an toàn với toạ độ âm đến `-2^31`.

In [17]:
class Solution:
    def findMinArrowShots(self, points: List[List[int]]) -> int:
        ans, last = 0, float("-inf")
        for a, b in sorted(points, key=lambda p: p[1]):
            if a > last:
                ans += 1
                last = b
        return ans


sol = Solution()
for pts, expected in examples:
    r = sol.findMinArrowShots(pts)
    assert r == expected
    print(pts, "->", r)

[[10, 16], [2, 8], [1, 6], [7, 12]] -> 2
[[1, 2], [3, 4], [5, 6], [7, 8]] -> 4
[[1, 2], [2, 3], [3, 4], [4, 5]] -> 2


## 10. Vì sao thuật toán đúng

**Cận dưới (không thể ít hơn).** Các quả mà thuật toán "bắn mũi tên mới" cho chúng — gọi là `E1, E2, ..., Em` theo thứ tự — **đôi một không chung điểm**: khi bắn cho `Ei` tại `last = bi` thì `E(i+1)` có `a > bi`, nên `E(i+1)` bắt đầu sau khi `Ei` đã kết thúc; vì đầu phải tăng dần nên mọi cặp đều rời nhau. Mỗi mũi tên nổ **nhiều nhất một** quả trong họ này, nên **cần ít nhất `m` mũi tên**.

**Cận trên (đủ `m` mũi tên).** Mọi quả `[a, b]` bị nổ: nếu thuật toán bắn cho nó thì hiển nhiên; nếu không thì `a <= last` với `last` là hoành độ mũi tên gần nhất; và quả này có `b >= last` (vì sắp theo đầu phải, `b >=` đầu phải của quả vừa được bắn `= last`), nên `a <= last <= b`: bị nổ.

Cận dưới = cận trên = `m` nên **thuật toán tối ưu**. Đây cũng chính là lập luận hoán đổi ở mục 6 viết dưới dạng quy nạp.

Ta kiểm chứng bằng oracle độc lập: trên hàng trăm input nhỏ, mọi cách = vét cạn; thêm nữa, các mũi tên tìm được **thật sự nổ hết** các quả bóng (kiểm tra bằng `covers_all`), và nếu **bỏ đi bất kỳ mũi tên nào** thì có quả không nổ (không thừa).

In [18]:
def arrow_positions(points):
    """Trả về danh sách vị trí các mũi tên (theo Cách 2) thay vì chỉ số lượng."""
    arrows, last = [], float("-inf")
    for a, b in sorted(points, key=lambda p: p[1]):
        if a > last:
            arrows.append(b)
            last = b
    return arrows


rng = random.Random(4520)
for _ in range(500):
    n = rng.randint(1, 8)
    pts = []
    for _ in range(n):
        a = rng.randint(-6, 8)
        pts.append([a, a + rng.randint(1, 7)])
    arrows = arrow_positions(pts)
    assert covers_all(arrows, pts)                       # đủ
    for i in range(len(arrows)):                         # không thừa: bỏ mũi nào cũng có quả không nổ
        assert not covers_all(arrows[:i] + arrows[i + 1:], pts)
    assert len(arrows) == min_arrows_bruteforce(pts)     # tối ưu (so với vét cạn)
print("500 input nhỏ: tập mũi tên đủ, không thừa, và đúng bằng số tối thiểu của vét cạn.")

500 input nhỏ: tập mũi tên đủ, không thừa, và đúng bằng số tối thiểu của vét cạn.


## 11. Phân tích Time / Space Complexity

Với `n = len(points)`:

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Vét cạn | `O(C(2n, k) · n · k)` (mũ) | `O(n)` | chỉ làm oracle `n ≤ 8` |
| 2. Sắp theo đầu phải | `O(n log n)` | `O(n)` | cách nộp bài |
| 3. Sắp theo đầu trái + giao nhóm | `O(n log n)` | `O(n)` | cùng bản chất, góc nhìn "nhóm" |
| 4. Sweep line | `O(n log n)` | `O(n)` | sự kiện `2n`, tổng quát hoá tốt |
| 5. Đối ngẫu DP + bisect | `O(n log n)` | `O(n)` | cho thấy đối ngẫu; nặng hơn không cần thiết |

Cận dưới của bài (mô hình so sánh) là `Ω(n log n)` do cần sắp xếp; vậy `O(n log n)` là tối ưu.

## 12. Bộ test

### 12.1. Ví dụ của đề và đối chiếu các cách

In [19]:
METHODS = {
    "by_end": arrows_by_end,
    "by_start": arrows_by_start,
    "sweep": arrows_sweep,
    "duality_dp": arrows_by_duality_dp,
    "Solution": Solution().findMinArrowShots,
}

for pts, expected in examples:
    for name, fn in METHODS.items():
        assert fn([p[:] for p in pts]) == expected, (name, pts)
    assert min_arrows_bruteforce(pts) == expected
print("3 ví dụ của đề: tất cả các cách đều đúng.")

3 ví dụ của đề: tất cả các cách đều đúng.


### 12.2. Edge case

Một quả bóng; hai quả giống hệt; hai quả lồng nhau; hai quả chạm đúng một điểm; chuỗi chạm nhau; các đoạn lồng nhau; toạ độ âm; toạ độ cực biên `-2^31`, `2^31 − 1`.

In [20]:
LO, HI = -2**31, 2**31 - 1
edge_cases = [
    ([[1, 2]], 1),
    ([[1, 2], [1, 2]], 1),                              # trùng nhau
    ([[1, 10], [2, 3]], 1),                             # lồng nhau
    ([[1, 2], [2, 3]], 1),                              # chạm nhau tại 2
    ([[1, 2], [3, 4]], 2),                              # rời nhau
    ([[1, 2], [2, 3], [3, 4], [4, 5]], 2),              # chuỗi chạm nhau
    ([[1, 10], [2, 9], [3, 8], [4, 7]], 1),             # lồng nhiều tầng
    ([[-5, -3], [-4, -1], [0, 2]], 2),                  # toạ độ âm
    ([[-5, -3]], 1),                                    # toạ độ âm một quả (last = 0 sẽ sai)
    ([[LO, HI]], 1),
    ([[LO, 0], [0, HI]], 1),                            # chạm nhau tại 0
    ([[LO, -1], [0, HI]], 2),
    ([[LO, LO + 1], [HI - 1, HI]], 2),
    ([[3, 9], [7, 12], [3, 8], [6, 8], [9, 10], [2, 9], [0, 9], [3, 9], [0, 6], [2, 8]], 2),
]
for pts, expected in edge_cases:
    for name, fn in METHODS.items():
        assert fn([p[:] for p in pts]) == expected, (name, pts)
    if len(pts) <= 8:
        assert min_arrows_bruteforce(pts) == expected
print(f"{len(edge_cases)} edge case: tất cả các cách đều đúng.")

14 edge case: tất cả các cách đều đúng.


### 12.3. Phản ví dụ cho các cách làm sai

Tham lam kiểu **"bắn chỗ có nhiều quả bóng nhất rồi lặp lại"** nghe hợp lý nhưng **sai**. Ví dụ dưới đây (tìm được bằng tìm kiếm có seed) cho thấy: điểm `x = 4` nổ **4** quả nhưng phá hỏng lời giải tối ưu.

In [21]:
def max_coverage_greedy(points):
    """SAI: mỗi lần bắn tại hoành độ nổ nhiều quả (còn lại) nhất."""
    rem = [tuple(p) for p in points]
    cands = sorted({x for a, b in points for x in (a, b)})
    shots = 0
    while rem:
        x = max(cands, key=lambda x: (sum(a <= x <= b for a, b in rem), -x))
        rem = [(a, b) for a, b in rem if not (a <= x <= b)]
        shots += 1
    return shots


counter = [[1, 3], [2, 4], [3, 4], [4, 5], [4, 6], [5, 6]]
print("tham lam 'nhiều bóng nhất':", max_coverage_greedy(counter))
print("đáp án đúng (vét cạn)    :", min_arrows_bruteforce(counter))
print("Cách 2                   :", arrows_by_end(counter))
assert max_coverage_greedy(counter) == 3 and min_arrows_bruteforce(counter) == 2 == arrows_by_end(counter)

tham lam 'nhiều bóng nhất': 3
đáp án đúng (vét cạn)    : 2
Cách 2                   : 2


Giải thích: điểm `x = 4` nằm trong `[2,4], [3,4], [4,5], [4,6]` (4 quả) nên bị chọn đầu tiên, nhưng còn `[1,3]` và `[5,6]` ở hai phía **không** chung điểm → thêm 2 mũi → tổng 3. Lời giải tối ưu dùng `x = 3` (nổ `[1,3],[2,4],[3,4]`) và `x = 5` (nổ `[4,5],[4,6],[5,6]`) → 2 mũi.

### 12.4. Đối chiếu ngẫu nhiên (có seed) với oracle vét cạn

Input nhỏ (`n ≤ 7`), toạ độ trong `[-6, 9]`, luôn `x_start < x_end` đúng ràng buộc. Mọi cách phải bằng vét cạn.

In [22]:
rng = random.Random(45200)
for trial in range(700):
    n = rng.randint(1, 7)
    pts = []
    for _ in range(n):
        a = rng.randint(-6, 8)
        pts.append([a, a + rng.randint(1, 6)])
    expected = min_arrows_bruteforce(pts)
    for name, fn in METHODS.items():
        assert fn([p[:] for p in pts]) == expected, (name, pts, expected)
    # sắp thứ tự đầu vào khác đi thì đáp án không đổi
    shuffled = pts[:]
    rng.shuffle(shuffled)
    assert arrows_by_end(shuffled) == expected
print("700 input ngẫu nhiên nhỏ: mọi cách = vét cạn, và bất biến với thứ tự đầu vào.")

700 input ngẫu nhiên nhỏ: mọi cách = vét cạn, và bất biến với thứ tự đầu vào.


### 12.5. Đối chiếu các cách nhanh với nhau trên input lớn (không có vét cạn)

`n` đến `2000`, toạ độ trong khoảng rất rộng (đến `±2^31`) lẫn khoảng hẹp (nhiều trùng lặp/chạm nhau). Các cách `O(n log n)` phải khớp nhau, và tập mũi tên của Cách 2 phải thật sự nổ hết các quả.

In [23]:
rng = random.Random(452452)
for trial in range(60):
    n = rng.randint(1, 2000)
    span = rng.choice([10, 100, 10**4, 2**31 - 1])
    pts = []
    for _ in range(n):
        a = rng.randint(-span, span - 1)
        b = rng.randint(a + 1, min(span, a + rng.choice([1, 3, 10, 2 * span])))
        pts.append([a, b])
    answers = {name: fn([p[:] for p in pts]) for name, fn in METHODS.items()}
    assert len(set(answers.values())) == 1, (trial, answers)
    arrows = arrow_positions(pts)
    assert len(arrows) == answers["by_end"]
    assert all(any(a <= x <= b for x in arrows) for a, b in pts)
print("60 input vừa/lớn: mọi cách khớp nhau; tập mũi tên nổ hết các quả.")

60 input vừa/lớn: mọi cách khớp nhau; tập mũi tên nổ hết các quả.


### 12.6. Hiệu năng ở giới hạn `n = 10^5`

In [24]:
rng = random.Random(77)
n = 10**5
big = []
for _ in range(n):
    a = rng.randint(-2**31, 2**31 - 2)
    b = rng.randint(a + 1, min(2**31 - 1, a + rng.choice([10, 10**6, 10**9])))
    big.append([a, b])

results = {}
for name, fn in METHODS.items():
    t0 = time.perf_counter()
    results[name] = fn([p[:] for p in big])
    print(f"{name:11s} {time.perf_counter() - t0:.3f}s  -> {results[name]}")
assert len(set(results.values())) == 1

by_end      0.051s  -> 36982


by_start    0.091s  -> 36982


sweep       0.172s  -> 36982


duality_dp  0.147s  -> 36982
Solution    0.060s  -> 36982


## 13. Các lỗi người mới hay gặp

### Lỗi 1: Sắp theo đầu trái nhưng dùng logic của "sắp theo đầu phải"

Quét theo đầu trái mà chỉ đặt `last = b` của quả **đầu tiên** của nhóm (không thu hẹp giao) thì sai.

In [25]:
def wrong_sort_by_start(points):
    ans, last = 0, float("-inf")
    for a, b in sorted(points):            # SAI: sắp theo đầu trái nhưng giữ nguyên logic cũ
        if a > last:
            ans += 1
            last = b
    return ans


pts = [[1, 10], [2, 3], [4, 5]]
print("sai :", wrong_sort_by_start(pts))
print("đúng:", arrows_by_end(pts), "(mũi ở 3 nổ [1,10] và [2,3]; mũi ở 5 nổ [4,5])")
assert wrong_sort_by_start(pts) == 1 and arrows_by_end(pts) == 2 == min_arrows_bruteforce(pts)

sai : 1
đúng: 2 (mũi ở 3 nổ [1,10] và [2,3]; mũi ở 5 nổ [4,5])


**Vì sao sai:** sau khi bắn tại `last = 10` cho `[1,10]`, ta nghĩ `[2,3]` và `[4,5]` đều bị nổ — nhưng mũi tên chỉ ở **một** điểm `10` mà `[2,3]` và `[4,5]` không chứa `10`. **Đúng:** sắp theo **đầu phải** (Cách 2), hoặc sắp theo đầu trái **và thu hẹp `hi = min(hi, b)`** (Cách 3).

### Lỗi 2: Dùng `>=` thay cho `>` (quên rằng chạm nhau cũng nổ chung)

In [26]:
def wrong_ge(points):
    ans, last = 0, float("-inf")
    for a, b in sorted(points, key=lambda p: p[1]):
        if a >= last:                       # SAI: chạm nhau (a == last) vẫn cùng một mũi tên
            ans += 1
            last = b
    return ans


pts = [[1, 2], [2, 3], [3, 4], [4, 5]]
print("sai :", wrong_ge(pts), "| đúng:", arrows_by_end(pts))
assert wrong_ge(pts) == 4 and arrows_by_end(pts) == 2

sai : 4 | đúng: 2


**Vì sao sai:** đoạn **đóng** `[1,2]` và `[2,3]` cùng chứa `x = 2`, nên một mũi tên đủ. Điều kiện bắn mũi mới phải là `a > last` (**nghiêm ngặt**).

### Lỗi 3: Khởi tạo `last = 0` (hoặc một số cụ thể)

In [27]:
def wrong_last_zero(points):
    ans, last = 0, 0                        # SAI: toạ độ có thể âm
    for a, b in sorted(points, key=lambda p: p[1]):
        if a > last:
            ans += 1
            last = b
    return ans


pts = [[-5, -3], [-2, -1]]
print("sai :", wrong_last_zero(pts), "| đúng:", arrows_by_end(pts))
assert wrong_last_zero(pts) == 0 and arrows_by_end(pts) == 2

sai : 0 | đúng: 2


**Vì sao sai:** với toạ độ âm, `a > 0` sai ngay từ quả đầu nên không bắn mũi nào. **Đúng:** `float("-inf")` (hoặc dùng cờ "chưa bắn").

### Lỗi 4: So sánh bằng phép trừ gây tràn số (ở ngôn ngữ có số nguyên 32-bit)

Trong C++/Java, `sort` với comparator `a[1] - b[1]` **tràn số** khi toạ độ ở biên `±2^31`. Python không tràn (số nguyên lớn tuỳ ý) nhưng bạn nên hiểu lý do; ta mô phỏng số nguyên 32-bit:

In [28]:
def to_int32(x):
    x &= 0xFFFFFFFF
    return x - (1 << 32) if x >= (1 << 31) else x


a_end, b_end = 2**31 - 1, -2**31 + 1
print("a - b trong số học thật  :", a_end - b_end, "(dương: a lớn hơn b)")
print("a - b mô phỏng int32     :", to_int32(a_end - b_end), "(âm: comparator trả lời SAI)")
assert a_end - b_end > 0 and to_int32(a_end - b_end) < 0
# Cách an toàn: so sánh trực tiếp  a[1] < b[1]  (hoặc key=lambda p: p[1] như ở Python)

a - b trong số học thật  : 4294967294 (dương: a lớn hơn b)
a - b mô phỏng int32     : -2 (âm: comparator trả lời SAI)


**Vì sao sai:** kết quả phép trừ vượt phạm vi `int` bị quấn vòng, đổi dấu. **Đúng:** so sánh trực tiếp (`a[1] < b[1]`), hoặc dùng `key`.

### Lỗi 5: Tham lam "bắn chỗ có nhiều bóng nhất"

Đã minh hoạ ở mục 12.3: `max_coverage_greedy` cho `3` thay vì `2`. Tham lam đúng là **tham lam theo đầu phải** (có chứng minh hoán đổi), không phải theo "mật độ".

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- **"Ít điểm nhất để chạm/phủ mọi đoạn"** (interval point cover) = **bài đoạn + tham lam**: sắp theo **đầu phải**, bắn tại đầu phải, bỏ mọi đoạn bị nổ.
- **Đoạn đóng hay mở?** Đoạn đóng: chạm nhau vẫn chung điểm → dùng `a > last` để mở mũi mới. Đoạn "nửa mở/ mở": dùng `a >= last`.
- **Họ bài "Interval + sắp xếp + quét"**: 56 Merge Intervals, 435 Non-overlapping Intervals, 253 Meeting Rooms II, 452 (bài này). Thường là: sắp theo đầu trái (gộp/đếm chồng) hoặc theo đầu phải (chọn tối đa/ phủ ít nhất).
- **Đối ngẫu**: số điểm phủ tối thiểu = số đoạn rời nhau tối đa (với đoạn trên trục số) — cho phép kiểm tra chéo hai bài.
- **Kiểm tra bằng oracle vét cạn** khi chỉ có thuật toán tham lam: tham lam dễ "nghe hợp lý mà sai" (mục 12.3).
- Toạ độ lớn đến `2^31`: không so sánh bằng phép trừ ở C++/Java; Python thì thoải mái.

## 15. Biến thể của bài

1. **Đoạn mở** (mũi tên chỉ nổ bóng khi `x_start < x < x_end`): thay `a > last` bằng `a >= last` (xem bài tập 5a).
2. **Non-overlapping Intervals** (435): ít đoạn nhất cần **xoá** để các đoạn còn lại không chồng nhau = `n −` số đoạn rời nhau tối đa; cho phép chạm nhau (xem bài tập 5b).
3. **Meeting Rooms II** (253): số phòng họp tối thiểu = độ sâu chồng tối đa; dùng sweep line như Cách 4 nhưng đếm thay vì xoá `active`.
4. **Merge Intervals** (56): gộp các đoạn chung điểm — gần với Cách 3 (giữ nhóm).
5. **Mũi tên có độ dài hữu hạn / có chi phí**: bài toán tổng quát (đặt ít "vòng phủ" nhất) cần DP.

Ví dụ nhỏ biến thể 3: độ sâu chồng tối đa bằng sweep line:

In [29]:
def max_overlap(points):
    events = []
    for a, b in points:
        events.append((a, 0))        # bắt đầu trước kết thúc khi cùng toạ độ (đoạn đóng)
        events.append((b, 1))
    events.sort()
    cur = best = 0
    for _, kind in events:
        cur += 1 if kind == 0 else -1
        best = max(best, cur)
    return best


print(max_overlap([[0, 30], [5, 10], [15, 20]]))       # 2
print(max_overlap([[1, 2], [2, 3]]))                   # 2 vì chạm nhau tại 2 (đoạn đóng)
assert max_overlap([[0, 30], [5, 10], [15, 20]]) == 2 and max_overlap([[1, 2], [2, 3]]) == 2

2
2


## 16. So sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. Vét cạn | thử mọi tập mũi tên | mũ | `O(n)` | oracle kiểm chứng `n ≤ 8` |
| 2. Sắp theo đầu phải | tham lam, bắn tại đầu phải | `O(n log n)` | `O(n)` | **nộp bài**, ngắn nhất |
| 3. Sắp theo đầu trái + giao | gom nhóm chung điểm | `O(n log n)` | `O(n)` | khi nghĩ theo "nhóm" (như Merge Intervals) |
| 4. Sweep line | sự kiện bắt đầu/kết thúc | `O(n log n)` | `O(n)` | tổng quát hoá (đếm chồng, nhiều ràng buộc) |
| 5. Đối ngẫu DP + bisect | số đoạn rời nhau tối đa | `O(n log n)` | `O(n)` | hiểu đối ngẫu; kiểm tra chéo |

Quy tắc chọn: nộp bài dùng Cách 2; cần mở rộng → Cách 4; muốn kiểm tra chéo → Cách 5.

## 17. 5 bài tập vận dụng (dễ → khó)

**Bài 1 (đoán output).** Không chạy code, đoán: `arrows_by_end([[1, 5]])`, `arrows_by_end([[1, 3], [3, 5]])`, `arrows_by_end([[1, 3], [4, 5]])`, `sorted([[10, 16], [2, 8], [1, 6]], key=lambda p: p[1])`, `share_point([1, 5], [5, 9])`.

**Bài 2 (tính tay).** Với `points = [[3,9],[7,12],[3,8],[6,8],[9,10],[2,9],[0,9],[3,9],[0,6],[2,8]]`: (a) sắp theo đầu phải và viết ra danh sách; (b) chạy tay thuật toán Cách 2, ghi từng mũi tên (vị trí và các quả nó nổ). Đáp án là bao nhiêu mũi tên?

**Bài 3 (điền TODO).** Hoàn thành `arrows_todo(points)` theo Cách 2.

In [30]:
def arrows_todo(points):
    # TODO 1: ans = 0; last = float("-inf")
    # TODO 2: duyệt (a, b) trong points đã sắp theo đầu phải: sorted(points, key=lambda p: p[1])
    # TODO 3: nếu a > last: ans += 1; last = b
    # TODO 4: return ans
    pass

# KHÔNG gọi hàm TODO ở đây để khi Run All không bị lỗi (TODO chưa điền nên kết quả chưa đúng).

**Bài 4 (tự viết lại).** Viết `arrow_plan(points)` trả về **danh sách các cặp** `(vị_trí_mũi_tên, [các chỉ số quả bóng nó nổ ĐẦU TIÊN])` sao cho mỗi quả bóng thuộc đúng một mũi tên (mũi tên bắn đầu tiên chạm nó). Kiểm tra: gộp mọi chỉ số được đủ `0..n-1`, và số mũi tên bằng đáp án.

**Bài 5 (biến thể / mini challenge).** (a) **Đoạn mở**: mũi tên nổ bóng khi `x_start < x < x_end` (toạ độ nguyên, mũi tên có thể bắn ở **số thực**). Viết `arrows_open(points)`. (b) Viết `min_removals_non_overlapping(intervals)` (bài 435: quy ước `[1,2]` và `[2,3]` **không chồng**) và kiểm tra bằng vét cạn.

## 18. Gợi ý 3 mức cho bài tập

- **Mức 1 (nhẹ):** Bài 1: `1`, `1` (chạm nhau tại 3), `2`, `[[1, 6], [2, 8], [10, 16]]`, `True`. Bài 3: bám sát mã Cách 2 mục 7.
- **Mức 2 (vừa):** Bài 2: sắp theo đầu phải được `[0,6],[3,8],[6,8],[2,8],[3,9],[2,9],[0,9],[3,9],[9,10],[7,12]`. Mũi 1 tại `6` (quả `[0,6]`): mọi quả có đầu trái `<= 6` đều nổ cho tới khi gặp `[9,10]` có `9 > 6` → mũi 2 tại `10`; còn `[7,12]` có `7 <= 10` nên nổ. Tổng `2`. Bài 4: trong vòng lặp, khi bắn mới tạo nhóm mới; mỗi quả được gán vào nhóm hiện tại (quả có `a <= last`).
- **Mức 3 (sâu):** Bài 5(a): với đoạn mở `(a, b)`, hai đoạn `(1,2)` và `(2,3)` **không** có chung điểm (nên cần 2 mũi): điều kiện mở mũi mới là `a >= last`. 5(b): đáp án `= n − (số đoạn rời nhau tối đa)` với "rời nhau" nghĩa `a >= last`; sắp theo đầu phải, tham lam.

## 19. Đáp án tham khảo cho bài tập

Chạy kiểm tra bằng `assert`.

In [31]:
# Bài 1
assert arrows_by_end([[1, 5]]) == 1
assert arrows_by_end([[1, 3], [3, 5]]) == 1
assert arrows_by_end([[1, 3], [4, 5]]) == 2
assert sorted([[10, 16], [2, 8], [1, 6]], key=lambda p: p[1]) == [[1, 6], [2, 8], [10, 16]]
assert share_point([1, 5], [5, 9]) is True

# Bài 2
pts2 = [[3, 9], [7, 12], [3, 8], [6, 8], [9, 10], [2, 9], [0, 9], [3, 9], [0, 6], [2, 8]]
sorted2 = sorted(pts2, key=lambda p: p[1])
print("Bài 2 (a):", sorted2)
arrows2 = arrow_positions(pts2)
print("Bài 2 (b): mũi tên tại", arrows2)
assert arrows2 == [6, 10] and arrows_by_end(pts2) == 2
assert covers_all(arrows2, pts2)
assert sorted2 == [[0, 6], [3, 8], [6, 8], [2, 8], [3, 9], [2, 9], [0, 9], [3, 9], [9, 10], [7, 12]]


# Bài 3
def arrows_todo_solution(points):
    ans, last = 0, float("-inf")
    for a, b in sorted(points, key=lambda p: p[1]):
        if a > last:
            ans += 1
            last = b
    return ans


for pts, expected in examples + edge_cases:
    assert arrows_todo_solution(pts) == expected


# Bài 4
def arrow_plan(points):
    order = sorted(range(len(points)), key=lambda i: points[i][1])
    plan = []
    last = float("-inf")
    for i in order:
        a, b = points[i]
        if a > last:
            plan.append((b, [i]))
            last = b
        else:
            plan[-1][1].append(i)
    return plan


rng = random.Random(99)
for _ in range(200):
    n = rng.randint(1, 8)
    pts = []
    for _ in range(n):
        a = rng.randint(-6, 8)
        pts.append([a, a + rng.randint(1, 6)])
    plan = arrow_plan(pts)
    assert len(plan) == arrows_by_end(pts)
    assert sorted(i for _, group in plan for i in group) == list(range(n))
    for x, group in plan:
        assert all(pts[i][0] <= x <= pts[i][1] for i in group)      # mũi tên thật sự nổ mọi quả trong nhóm


# Bài 5a: đoạn mở (a, b): chung điểm khi max(a) < min(b)
def arrows_open(points):
    ans, last = 0, float("-inf")
    for a, b in sorted(points, key=lambda p: p[1]):
        if a >= last:
            ans += 1
            last = b
    return ans


def arrows_open_bruteforce(points):
    # mũi tên có thể ở số thực: ứng viên là trung điểm giữa các toạ độ liền kề
    coords = sorted({x for a, b in points for x in (a, b)})
    cands = [(coords[i] + coords[i + 1]) / 2 for i in range(len(coords) - 1)]
    for k in range(1, len(points) + 1):
        for combo in combinations(cands, k):
            if all(any(a < x < b for x in combo) for a, b in points):
                return k


rng = random.Random(5)
for _ in range(300):
    n = rng.randint(1, 6)
    pts = []
    for _ in range(n):
        a = rng.randint(-4, 6)
        pts.append([a, a + rng.randint(1, 5)])
    assert arrows_open(pts) == arrows_open_bruteforce(pts), pts
assert arrows_open([[1, 2], [2, 3]]) == 2 and arrows_by_end([[1, 2], [2, 3]]) == 1


# Bài 5b: Non-overlapping Intervals (435)
def min_removals_non_overlapping(intervals):
    kept, last = 0, float("-inf")
    for a, b in sorted(intervals, key=lambda p: p[1]):
        if a >= last:                      # chạm nhau (a == last) vẫn KHÔNG chồng
            kept += 1
            last = b
    return len(intervals) - kept


def min_removals_bruteforce(intervals):
    n = len(intervals)
    for r in range(n + 1):
        for removed in combinations(range(n), r):
            keep = [intervals[i] for i in range(n) if i not in removed]
            if all(keep[i][1] <= keep[j][0] or keep[j][1] <= keep[i][0]
                   for i in range(len(keep)) for j in range(i + 1, len(keep))):
                return r


rng = random.Random(6)
for _ in range(300):
    n = rng.randint(1, 7)
    iv = []
    for _ in range(n):
        a = rng.randint(-4, 6)
        iv.append([a, a + rng.randint(1, 5)])
    assert min_removals_non_overlapping(iv) == min_removals_bruteforce(iv), iv
assert min_removals_non_overlapping([[1, 2], [2, 3], [3, 4], [1, 3]]) == 1
print("Tất cả đáp án bài tập đều đúng.")

Bài 2 (a): [[0, 6], [3, 8], [6, 8], [2, 8], [3, 9], [2, 9], [0, 9], [3, 9], [9, 10], [7, 12]]
Bài 2 (b): mũi tên tại [6, 10]
Tất cả đáp án bài tập đều đúng.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

**Template** (ít điểm nhất để chạm mọi đoạn đóng — sắp theo đầu phải)

```python
class Solution:
    def findMinArrowShots(self, points: List[List[int]]) -> int:
        ans, last = 0, float("-inf")           # last = hoành độ mũi tên gần nhất
        for a, b in sorted(points, key=lambda p: p[1]):   # sắp theo ĐẦU PHẢI
            if a > last:                       # chưa bị nổ (nghiêm ngặt: chạm nhau vẫn nổ chung)
                ans += 1
                last = b                       # bắn tại đầu phải
        return ans
```

**Checklist**

- [ ] Sắp theo **đầu phải** (nếu sắp theo đầu trái thì phải thu hẹp giao `hi = min(hi, b)`)?
- [ ] Đoạn **đóng**: dùng `a > last` (không phải `>=`)? Đoạn mở thì ngược lại?
- [ ] `last` khởi tạo `float("-inf")` (toạ độ có thể âm, đến `-2^31`)?
- [ ] Ở ngôn ngữ khác: không dùng `a[1] - b[1]` làm comparator (tràn số)?
- [ ] Không dùng tham lam "bắn chỗ nhiều bóng nhất"?
- [ ] Đã thử: 1 quả, hai quả trùng, lồng nhau, chạm nhau (`[1,2],[2,3]`), toạ độ âm, `±2^31`, `n = 10^5`?
- [ ] Skeleton đúng `class Solution` / `def findMinArrowShots(self, points: List[List[int]]) -> int`?